# Data-derived covariates (p2) -- control genes, RUVg factors, RDA, VIF

1. Control genes: the N detected genes least associated with phenotype (all samples, disease included) plus the
   published sex-marker genes (always; sex metadata is missing for many HC, so sex enters through genes)
2. Overlap of the control set with PanglaoDB hematopoietic cell types (Platelets = the Platelet Score genes)
3. RUVg factors W from HC only (`RUVSeq::RUVg` algorithm, checked against R), projected onto every sample; sex axis check
4. RDA with BIAS10: BIAS10 + W_1..k for every control-set size N; where the BIAS10 unique variance comes from
5. VIF: BIAS10 + W_1..k and depth + W_1..k, for every N and k
6. RDA of the compact set: depth + W_1..k, against BIAS10 alone
7. In-silico projection absorption: how much injected signal W takes up, and how much it spills onto other genes
8. Choosing N and k: every criterion of sections 1-7 in one figure each (`covariate_8*`) and one summary table

Everything here is fitted on all engine HC at once and is for choosing (control-gene rule, N, k) only.
The p2 provider refits W inside every CV fold / LOBO training set.
The h5ad `RUVg_Platelet_*` / `Proposed_Full_*` layers are per-study fits and are never used.

Outputs: `results/covariates/` (tables, W) and `results/Figures/covariate_<section>_*.png`.

In [21]:
import hashlib
import subprocess
import sys
import tempfile
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
from scipy.stats import false_discovery_control, fisher_exact

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import FS_ANNOT_SM, apply_style
apply_style()

import _FinalModelingMethodImprovement.config as config
from _FinalModelingMethodImprovement.engine.validation.lobo_engine import _decode_cat, load_full_data
from _FinalModelingMethodImprovement.eval.insilico_sweep import cell_path, fold_spec, universe

OUT = config.COVARIATE_DIR
FIG_DIR = config.RESULTS_DIR / 'Figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DETECT_MIN = 0.95
MIN_BATCH_N = 10
N_CONTROLS = [100, 300, 500, 1000]
N_MAIN = 500
K_MAIN = 6
K_MAX = 10
SEX_GENES = ['RPS4Y1', 'EIF1AY', 'DDX3Y', 'KDM5D']
DEPTH = 'log(Total Reads)'
TAG = f'det{DETECT_MIN}_N{"-".join(map(str, N_CONTROLS))}_K{K_MAX}'
pd.set_option('display.width', 250)

In [22]:
data = load_full_data()
genes = list(data['gene_col'])
Y = data['Y'][:, [data['gene_col'][g] for g in genes]]
lib = data['Y'].sum(1)
del data['Y']
logcpm = np.log2(Y / lib[:, None] * 1e6 + 1)
col = {g: i for i, g in enumerate(genes)}
with h5py.File(config.H5AD_PATH, 'r') as f:
    sym_of = dict(zip(f['var/_index'][()].astype(str), _decode_cat(f['var/GeneName'])))
    chrom_of = dict(zip(f['var/_index'][()].astype(str), _decode_cat(f['var/Chromosome'])))
    obs_names = f['obs/_index'][()].astype(str)
    sex = pd.Series(_decode_cat(f['obs/sex']), index=obs_names).loc[data['names']].values
    sex_score = pd.Series(f['obs/Sex Score'][()], index=obs_names).loc[data['names']].values
modeled = set(pd.read_csv(config.ENGINE_MIXED_DIR / 'training_summary.csv', usecols=['gene'])['gene'])
batch, pheno = data['batch'], data['phenotype']
hc_fit = data['is_hc'] & ~np.isin(batch, list(data['small_hc_batches']))
print(f'samples {len(batch)}, HC used for W {hc_fit.sum()}, protein-coding genes {len(genes)}, modeled {len(modeled)}')

samples 1606, HC used for W 679, protein-coding genes 20097, modeled 20097


## 1. Control genes

Detection: count > 0 in at least `DETECT_MIN` of the samples of every batch with >= `MIN_BATCH_N` samples.
Phenotype association: per gene, F-test of `logCPM ~ batch + phenotype` against `logCPM ~ batch` over all samples,
so only within-batch phenotype contrasts count. Control set = the N detected, modeled genes with the largest p (equivalently the smallest phenotype partial R^2,
`pheno_r2` = share of the within-batch variance explained by phenotype, since every gene has the same df).

Sex genes (always added): the chrY genes of the tissue- and platform-independent sex-marker panel of Staedtler et al. 2013
(Biomarkers 18:436, doi:10.3109/1354750X.2013.811538: RPS4Y1, EIF1AY, DDX3Y, KDM5D; the fifth, XIST, is not protein-coding
and so not in this matrix). Fixed in advance, not selected from this data; their male / female detection is printed as a check.
The detection rule alone removes them (zero in females), so W could never carry a sex axis otherwise.

Limitation (accepted): a large group-level p is weak evidence of no disease association -- most batches hold a
single phenotype, the within-batch contrasts come from few batches, and non-significance is not absence of effect.

In [23]:
def basis(X):
    u, s, _ = np.linalg.svd(X, full_matrices=False)
    return u[:, s > s[0] * 1e-10]


def rss(Q, Yc):
    return (Yc ** 2).sum(0) - ((Q.T @ Yc) ** 2).sum(0)


STATS_CACHE = OUT / 'control_gene_stats.csv'
if STATS_CACHE.is_file():
    stats = pd.read_csv(STATS_CACHE, index_col='gene')
else:
    big = pd.Series(batch).value_counts().loc[lambda v: v >= MIN_BATCH_N].index
    det = np.min([(Y[batch == b] > 0).mean(0) for b in big], axis=0)
    B = pd.get_dummies(batch).values.astype(float)
    P = pd.get_dummies(pheno).values.astype(float)
    Q0, Q1 = basis(B), basis(np.hstack([B, P]))
    df1, df2 = Q1.shape[1] - Q0.shape[1], len(batch) - Q1.shape[1]
    r0, r1 = rss(Q0, logcpm), rss(Q1, logcpm)
    with np.errstate(invalid="ignore", divide="ignore"):
        F = ((r0 - r1) / df1) / (r1 / df2)
        r2p = (r0 - r1) / r0
    stats = pd.DataFrame(dict(symbol=[sym_of.get(g, '') for g in genes], min_batch_detect=det,
                              mean_logcpm=logcpm.mean(0), F=F, p=f_dist.sf(F, df1, df2), pheno_r2=r2p,
                              modeled=[g in modeled for g in genes]), index=pd.Index(genes, name='gene'))
    stats.to_csv(STATS_CACHE)

mix = pd.crosstab(batch, pheno == 'Healthy Control').loc[lambda t: (t > 0).all(1)]
print(f'batches with both HC and disease: {len(mix)} ({mix.values.sum()} samples)')
cand = stats[(stats['min_batch_detect'] >= DETECT_MIN) & stats['modeled']].sort_values('p', ascending=False)
male, female = hc_fit & (sex == 'male'), hc_fit & (sex == 'female')
sex_genes = [g for g in genes if sym_of.get(g) in SEX_GENES and chrom_of.get(g) in ('chrY', 'Y')]
assert sorted(sym_of[g] for g in sex_genes) == sorted(SEX_GENES)
print(f"HC sex: {male.sum()} male, {female.sum()} female, {hc_fit.sum() - male.sum() - female.sum()} missing")
display(pd.DataFrame({sym_of[g]: dict(detect_known_male=(Y[male][:, col[g]] > 0).mean(), detect_known_female=(Y[female][:, col[g]] > 0).mean())
                      for g in sex_genes}).T.round(3))


def with_sex(c):
    return c + [g for g in sex_genes if g not in set(c)]


controls = {n: with_sex(cand.index[:n].tolist()) for n in N_CONTROLS}
TAG = f"{TAG}_{hashlib.md5('|'.join(cand.index[:max(N_CONTROLS)].tolist() + sex_genes).encode()).hexdigest()[:8]}"
print(f'cache tag: {TAG}')
for n, c in controls.items():
    (OUT / f'control_genes_N{n}.txt').write_text('\n'.join(c))
print(f'detected + modeled: {len(cand)} of {len(stats)}')
display(pd.DataFrame({t: dict(n_genes=int(((stats['min_batch_detect'] >= t) & stats['modeled']).sum()),
                              n_p_gt_0_05=int(((stats['min_batch_detect'] >= t) & stats['modeled'] & (stats['p'] > 0.05)).sum()))
                      for t in (0.5, 0.7, 0.8, 0.9, 0.95)}).T.rename_axis('DETECT_MIN'))
summ = {n: dict(min_p_in_set=stats.loc[c, 'p'].min(), median_pheno_r2=stats.loc[c, 'pheno_r2'].median(),
                max_pheno_r2=stats.loc[c, 'pheno_r2'].max(), median_mean_logcpm=stats.loc[c, 'mean_logcpm'].median())
        for n, c in controls.items()}
summ['all detected'] = dict(min_p_in_set=np.nan, median_pheno_r2=cand['pheno_r2'].median(), max_pheno_r2=cand['pheno_r2'].max(),
                            median_mean_logcpm=cand['mean_logcpm'].median())
display(pd.DataFrame(summ).T)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
bins = np.linspace(0, cand['pheno_r2'].max(), 60)
axes[0].hist(cand['pheno_r2'], bins=bins, color='0.75', label='detected')
for n, color in zip(sorted(N_CONTROLS, reverse=True), ['#E69F00', '#0072B2', '#000000']):
    axes[0].hist(stats.loc[controls[n], 'pheno_r2'], bins=bins, histtype='step', color=color, label=f'control N={n}')
axes[0].set_xlabel('Phenotype partial R2 (within batch)'); axes[0].set_ylabel('Genes')
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM)
axes[1].scatter(cand['mean_logcpm'], -np.log10(cand['p'].clip(lower=1e-300)), s=2, color='0.75', label='detected')
for n, color in zip(sorted(N_CONTROLS, reverse=True), ['#E69F00', '#0072B2', '#000000']):
    c = stats.loc[controls[n]]
    axes[1].scatter(c['mean_logcpm'], -np.log10(c['p']), s=2, color=color, label=f'control N={n}')
axes[1].set_xlabel('Mean log2 CPM'); axes[1].set_ylabel('-log10 p')
axes[1].legend(frameon=False, fontsize=FS_ANNOT_SM, markerscale=4)
fig.tight_layout()
fig.savefig(FIG_DIR / 'covariate_1_control_genes.png', dpi=300, bbox_inches='tight')
plt.show()

batches with both HC and disease: 26 (1399 samples)
HC sex: 114 male, 331 female, 234 missing


,detect_known_male,detect_known_female
KDM5D,1.0,0.051
DDX3Y,1.0,0.069
RPS4Y1,1.0,0.066
EIF1AY,1.0,0.042


cache tag: det0.95_N100-300-500-1000_K10_db59739b
detected + modeled: 1227 of 20097


,n_genes,n_p_gt_0_05
DETECT_MIN,,
0.50,7101,114
0.70,4661,52
0.80,3419,27
0.90,2018,7
0.95,1227,2


,min_p_in_set,median_pheno_r2,max_pheno_r2,median_mean_logcpm
100,1.694960e-10,0.044095,0.054917,6.696194
300,5.720430e-20,0.062430,0.085832,6.804466
500,2.262732e-28,0.078137,0.111027,6.674848
1000,9.009550e-47,0.110799,0.162033,6.407145
all detected,NaN,0.123406,0.225017,6.584789


/tmp/ipykernel_540966/3103593670.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Overlap with PanglaoDB hematopoietic cell types

Cell types of organ Blood / Immune system plus Hematopoietic stem cells (human markers).
Universe = detected, modeled genes. One-sided Fisher test (enrichment), BH across cell types x N.
`n_markers` = PanglaoDB human markers matched to a protein-coding gene; `n_detected` = those passing the detection rule
(the Fisher universe). Heatmap: cell types with >= 5 detected markers; label n = all markers of the set,
cell text = control genes in the set, colour = share of the detected markers.

In [24]:
pg = pd.read_csv(config.PANGLAO_TSV, sep='\t')
pg = pg[pg['species'].str.contains('Hs') & (pg['organ'].isin(['Blood', 'Immune system']) | (pg['cell type'] == 'Hematopoietic stem cells'))]
univ = set(cand['symbol']) - {''}
pc_sym = set(stats['symbol']) - {''}
n_markers = {ct: len(set(g['official gene symbol']) & pc_sym) for ct, g in pg.groupby('cell type')}
sets = {ct: set(g['official gene symbol']) & univ for ct, g in pg.groupby('cell type')}
rows = []
for n, c in controls.items():
    cs = (set(stats.loc[c, 'symbol']) - {''}) & univ
    for ct, s in sets.items():
        a = len(cs & s)
        tab = [[a, len(cs) - a], [len(s) - a, len(univ) - len(cs) - len(s) + a]]
        odds, p = fisher_exact(tab, alternative='greater')
        rows.append(dict(N=n, cell_type=ct, n_markers=n_markers[ct], n_detected=len(s), overlap=a,
                         frac_of_markers=a / max(n_markers[ct], 1), frac_of_detected=a / max(len(s), 1),
                         jaccard=a / len(cs | s), odds_ratio=odds, p=p))
ov = pd.DataFrame(rows)
ov['q'] = false_discovery_control(ov['p'])
ov.to_csv(OUT / 'control_panglao_overlap.csv', index=False)
display(ov[ov['N'] == N_MAIN].sort_values('p').round(4).head(15))

mat = ov.pivot(index='cell_type', columns='N', values='frac_of_detected')
q = ov.pivot(index='cell_type', columns='N', values='q').loc[mat.index]
cnt = ov.pivot(index='cell_type', columns='N', values='overlap').loc[mat.index]
size = ov.groupby('cell_type')[['n_detected', 'n_markers']].first()
order = mat[size['n_detected'].reindex(mat.index) >= 5][N_MAIN].sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(4.5, 0.28 * len(order) + 1))
im = ax.imshow(mat.loc[order].values, cmap='Greys', aspect='auto', vmin=0)
for i, ct in enumerate(order):
    for j, n in enumerate(mat.columns):
        v = mat.loc[ct, n]
        ax.text(j, i, f"{int(cnt.loc[ct, n])}{'*' if q.loc[ct, n] < 0.05 else ''}", ha='center', va='center',
                fontsize=FS_ANNOT_SM, color='white' if v > 0.5 else 'black')
ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels([f'N={n}' for n in mat.columns])
ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{ct} (n={size.loc[ct, 'n_markers']})" for ct in order], fontsize=FS_ANNOT_SM)
fig.colorbar(im, ax=ax, label='Fraction of detected markers in control set')
ax.set_title('Control set vs PanglaoDB\n(cell: overlapping genes, * BH q < 0.05)')
fig.savefig(FIG_DIR / 'covariate_2_panglao_overlap.png', dpi=300, bbox_inches='tight')
plt.show()

,N,cell_type,n_markers,n_detected,overlap,frac_of_markers,frac_of_detected,jaccard,odds_ratio,p,q
81,500,Platelets,126,37,30,0.2381,0.8108,0.0592,6.5653,0.0000,0.0
71,500,Mast cells,137,5,4,0.0292,0.8000,0.0080,5.8548,0.0925,1.0
67,500,Erythroid-like and erythroid precursor cells,70,21,11,0.1571,0.5238,0.0216,1.6129,0.1914,1.0
72,500,Megakaryocytes,47,10,5,0.1064,0.5000,0.0099,1.4586,0.3855,1.0
82,500,Red pulp macrophages,12,1,1,0.0833,1.0000,0.0020,inf,0.4075,1.0
77,500,Neutrophils,69,8,4,0.0580,0.5000,0.0079,1.4577,0.4232,1.0
70,500,Macrophages,127,14,6,0.0472,0.4286,0.0118,1.0916,0.5377,1.0
65,500,Eosinophils,27,2,1,0.0370,0.5000,0.0020,1.4549,0.6491,1.0
83,500,Reticulocytes,8,2,1,0.1250,0.5000,0.0020,1.4549,0.6491,1.0
87,500,T helper cells,60,2,1,0.0167,0.5000,0.0020,1.4549,0.6491,1.0


/tmp/ipykernel_540966/792719727.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. RUVg factors

`RUVSeq::RUVg(isLog=TRUE, center=TRUE)`: W = left singular vectors of the HC control matrix centred by the HC gene means.
Other samples: W_new = (x_ctrl - HC mean) V D^-1, which reproduces W exactly on the HC.
The R check compares Python and R W column by column (|r|, sign is arbitrary).
Sex enters only through the sex genes in the control set. To check that W carries it, the tables give the R^2 of the
known sex (known-sex HC) and of obs['Sex Score'] (all HC; mean log2 CPM of the same panel, written by
`eval/add_sex_score.py`) on W_1..k. Section 6 tests Sex Score as an extra covariate; all sex checks are plotted in 8c.

In [25]:
def ruvg_fit(x_ctrl_hc, k):
    m = x_ctrl_hc.mean(0)
    u, d, vt = np.linalg.svd(x_ctrl_hc - m, full_matrices=False)
    return dict(mean=m, v=vt[:k].T, d=d[:k], var_frac=d ** 2 / (d ** 2).sum())


def ruvg_transform(fit, x_ctrl):
    return (x_ctrl - fit['mean']) @ fit['v'] / fit['d']


fits, W = {}, {}
for n, c in controls.items():
    idx = [col[g] for g in c]
    fits[n] = ruvg_fit(logcpm[hc_fit][:, idx], K_MAX)
    W[n] = ruvg_transform(fits[n], logcpm[:, idx])
    pd.DataFrame(W[n], index=data['names'], columns=[f'W_{i + 1}' for i in range(K_MAX)]).assign(
        hc_fit=hc_fit, batch=batch, phenotype=pheno).to_csv(OUT / f'W_N{n}.csv')

R_CHECK = OUT / f'ruvg_r_check_N{N_MAIN}_{TAG}.csv'
rc = pd.read_csv(R_CHECK) if R_CHECK.is_file() else None
if rc is None or len(rc) != K_MAX:
    with tempfile.TemporaryDirectory() as tmp:
        idx = [col[g] for g in controls[N_MAIN]]
        pd.DataFrame(logcpm[hc_fit][:, idx].T).to_csv(f'{tmp}/x.csv', index=False)
        r = ('suppressMessages(library(RUVSeq)); x <- as.matrix(read.csv(commandArgs(TRUE)[1])); '
             f'w <- RUVg(x, seq_len(nrow(x)), k = {K_MAX}, isLog = TRUE, center = TRUE)$W; '
             'write.csv(w, commandArgs(TRUE)[2], row.names = FALSE)')
        subprocess.run([str(config.RUVSEQ_RSCRIPT), '-e', r, f'{tmp}/x.csv', f'{tmp}/w.csv'], check=True)
        w_r = pd.read_csv(f'{tmp}/w.csv').values
    rc = pd.DataFrame(dict(W=[f'W_{i + 1}' for i in range(K_MAX)],
                           abs_r=[abs(np.corrcoef(W[N_MAIN][hc_fit][:, i], w_r[:, i])[0, 1]) for i in range(K_MAX)],
                           max_abs_diff=[np.abs(np.abs(W[N_MAIN][hc_fit][:, i]) - np.abs(w_r[:, i])).max() for i in range(K_MAX)]))
    rc.to_csv(R_CHECK, index=False)
display(rc.round(6))

known = male | female
sx = male[known].astype(float)
sex_w = pd.DataFrame({n: {k: 1 - rss(basis(np.column_stack([np.ones(known.sum()), W[n][known][:, :k]])), (sx - sx.mean())[:, None])[0]
                             / ((sx - sx.mean()) ** 2).sum() for k in (1, 2, 3, 5, 6, 10)} for n in N_CONTROLS})
sex_w.index.name, sex_w.columns.name = 'k', 'N'
ss = sex_score[hc_fit] - sex_score[hc_fit].mean()
score_w = pd.DataFrame({n: {k: 1 - rss(basis(np.column_stack([np.ones(hc_fit.sum()), W[n][hc_fit][:, :k]])), ss[:, None])[0] / (ss ** 2).sum()
                            for k in (1, 2, 3, 5, 6, 10)} for n in N_CONTROLS})
score_w.index.name, score_w.columns.name = 'k', 'N'
print('R2 of known sex on W_1..k (known-sex HC):')
display(sex_w.round(3))
print('R2 of Sex Score on W_1..k (all HC):')
display(score_w.round(3))

,W,abs_r,max_abs_diff
0,W_1,1.0,0.0
1,W_2,1.0,0.0
2,W_3,1.0,0.0
3,W_4,1.0,0.0
4,W_5,1.0,0.0
5,W_6,1.0,0.0
6,W_7,1.0,0.0
7,W_8,1.0,0.0
8,W_9,1.0,0.0
9,W_10,1.0,0.0


R2 of known sex on W_1..k (known-sex HC):


N,100,300,500,1000
k,,,,
1,0.087,0.002,0.043,0.049
2,0.893,0.255,0.198,0.184
3,0.926,0.901,0.201,0.185
5,0.927,0.917,0.853,0.308
6,0.928,0.925,0.916,0.601
10,0.929,0.929,0.929,0.921


R2 of Sex Score on W_1..k (all HC):


N,100,300,500,1000
k,,,,
1,0.001,0.006,0.028,0.038
2,0.975,0.072,0.043,0.041
3,0.996,0.959,0.061,0.083
5,0.997,0.971,0.781,0.316
6,0.997,0.984,0.957,0.509
10,0.999,0.997,0.994,0.971


### 3b. Lower bound on N: bootstrap stability of W

RUV control genes must carry the unwanted variation (Gagnon-Bartsch & Speed 2012, Biostatistics 13:539,
doi:10.1093/biostatistics/kxr034; Risso et al. 2014, Nat Biotechnol 32:896, doi:10.1038/nbt.2931). Too few genes and W is
fitted noise. Stability = mean canonical correlation between W_1..k of the full HC fit and W_1..k of a fit on a bootstrap
resample of the HC (both evaluated on all HC; 1 = same subspace). Lower bound `N_LOWER` = the smallest N whose
5th percentile over resamples is >= `STAB_MIN` at k = `STAB_K`. The control set always includes the sex genes.
Plotted in 8b. Sampling noise only: a LOBO / CV training set holds fewer HC than the full fit, so its W is somewhat less stable.

In [26]:
STAB_NS = [25, 50, 100, 200, 300, 500, 750, 1000, len(cand)]
STAB_KS = [3, 6, 10]
STAB_K = 6
STAB_B = 100
STAB_MIN = 0.95
STAB_CACHE = OUT / f'subspace_stability_{TAG}_n{len(cand)}_B{STAB_B}.csv'


def canon_corr(a, b):
    return np.linalg.svd(basis(a - a.mean(0)).T @ basis(b - b.mean(0)), compute_uv=False)


if STAB_CACHE.is_file():
    stab = pd.read_csv(STAB_CACHE)
else:
    rng = np.random.default_rng(0)
    hc_rows = np.where(hc_fit)[0]
    boots = [rng.choice(hc_rows, len(hc_rows)) for _ in range(STAB_B)]
    rows = []
    for n in STAB_NS:
        idx = [col[g] for g in with_sex(cand.index[:n].tolist())]
        x = logcpm[hc_rows][:, idx]
        ref = ruvg_transform(ruvg_fit(x, K_MAX), x)
        for b, rb in enumerate(boots):
            wb = ruvg_transform(ruvg_fit(logcpm[rb][:, idx], K_MAX), x)
            for k in STAB_KS:
                cc = canon_corr(ref[:, :k], wb[:, :k])
                rows.append(dict(N=n, k=k, boot=b, mean_cc=cc.mean(), min_cc=cc.min()))
    stab = pd.DataFrame(rows)
    stab.to_csv(STAB_CACHE, index=False)

stab_summary = stab.groupby(['k', 'N'])['mean_cc'].describe(percentiles=[0.05])[['5%', '50%']].unstack('k')
display(stab_summary.round(4))
ok = stab_summary[('5%', STAB_K)] >= STAB_MIN
N_LOWER = ok.index[ok.values.argmax()] if ok.any() else None
print(f'N_LOWER (k={STAB_K}, 5th percentile >= {STAB_MIN}): {N_LOWER}; N_MAIN = {N_MAIN}')

5%                     50%                
k         3       6       10      3       6       10
N                                                   
25    0.9652  0.8374  0.9114  0.9874  0.9616  0.9799
50    0.9130  0.8501  0.8964  0.9652  0.9573  0.9557
100   0.9484  0.9747  0.8928  0.9975  0.9974  0.9655
200   0.9719  0.9814  0.9253  0.9968  0.9967  0.9830
300   0.8451  0.9807  0.9075  0.9910  0.9955  0.9825
500   0.9374  0.9819  0.9024  0.9738  0.9968  0.9852
750   0.8866  0.9745  0.9060  0.9891  0.9945  0.9815
1000  0.7744  0.9607  0.9082  0.9898  0.9917  0.9708
1227  0.6895  0.9613  0.9277  0.9950  0.9922  0.9821

N_LOWER (k=6, 5th percentile >= 0.95): 100; N_MAIN = 500


## 4. RDA with BIAS10 -- explained variance of HC expression

Y = HC log2 CPM of the modeled genes minus every control gene of the largest N (same Y for all N, no circularity).
Multivariate adjusted R^2 (`EDA/analysis/pipeline.py` definition). `W_k unique` = R^2(BIAS10 + W_k) - R^2(BIAS10);
`batch left` = R^2(covariates + Batch_ID) - R^2(covariates), the batch variance the covariates do not absorb.

Stacked bars, one per case (BIAS10 only, then BIAS10 + W_1..k): BIAS10 unique = R^2(BIAS10 + W) - R^2(W),
W unique = R^2(BIAS10 + W) - R^2(BIAS10), shared = the rest of R^2(BIAS10 + W); number on top = R^2(BIAS10 + W).

In [27]:
def adj_r2(Yc, X):
    Q = basis(np.column_stack([np.ones(len(Yc)), X]) if X is not None else np.ones((len(Yc), 1)))
    sst = (Yc ** 2).sum()
    r2 = 1 - rss(Q, Yc).sum() / sst
    n, p = len(Yc), Q.shape[1] - 1
    return max(0.0, 1 - (1 - r2) * (n - 1) / (n - p - 1))


RDA_CACHE = OUT / f'rda_hc_{TAG}.csv'
excl = set(controls[max(N_CONTROLS)])
Yc = logcpm[hc_fit][:, [col[g] for g in genes if g in modeled and g not in excl]]
Yc = Yc - Yc.mean(0)
Xb = data['X_raw'][hc_fit]
Xb = (Xb - Xb.mean(0)) / Xb.std(0)
Bt = pd.get_dummies(batch[hc_fit]).values.astype(float)
rda = pd.read_csv(RDA_CACHE) if RDA_CACHE.is_file() else None
if rda is None:
    base = dict(bias=adj_r2(Yc, Xb), batch=adj_r2(Yc, Bt))
    rows = [dict(N=0, k=0, W=0.0, bias_W=base['bias'], bias=base['bias'], batch=base['batch'],
                 batch_left=adj_r2(Yc, np.hstack([Xb, Bt])) - base['bias'])]
    for n in N_CONTROLS:
        Wh = W[n][hc_fit]
        for k in range(1, K_MAX + 1):
            bw = adj_r2(Yc, np.hstack([Xb, Wh[:, :k]]))
            rows.append(dict(N=n, k=k, W=adj_r2(Yc, Wh[:, :k]), bias_W=bw, bias=base['bias'], batch=base['batch'],
                             batch_left=adj_r2(Yc, np.hstack([Xb, Wh[:, :k], Bt])) - bw))
    rda = pd.DataFrame(rows)
    rda['W_unique'] = rda['bias_W'] - rda['bias']
    rda['bias_unique'] = rda['bias_W'] - rda['W']
    rda.to_csv(RDA_CACHE, index=False)
display(rda.round(4))

PARTS = [('BIAS10 unique', '#D55E00'), ('Shared', '#CC79A7'), ('W unique', '#0072B2'),
         ('Batch left over', '#E69F00'), ('Unexplained', '#E5E5E5')]
b0 = rda[rda['N'] == 0].assign(k=0)
fig, axes = plt.subplots(1, len(N_CONTROLS), figsize=(4.2 * len(N_CONTROLS), 4), sharey=True, squeeze=False)
for ax, n in zip(axes[0], N_CONTROLS):
    d = pd.concat([b0, rda[rda['N'] == n]]).set_index('k')
    parts = pd.DataFrame({'BIAS10 unique': d['bias_unique'], 'Shared': d['bias_W'] - d['bias_unique'] - d['W_unique'],
                          'W unique': d['W_unique'], 'Batch left over': d['batch_left']})
    parts['Unexplained'] = 1 - parts.sum(1)
    bottom = np.zeros(len(parts))
    for name, color in PARTS:
        ax.bar(range(len(parts)), 100 * parts[name], bottom=bottom, color=color, edgecolor='white', lw=0.4, label=name)
        bottom += 100 * parts[name].values
    for x, (v, top) in enumerate(zip(d['bias_W'], d['bias_W'] + d['batch_left'])):
        ax.text(x, 100 * top + 1, f'{100 * v:.0f}', ha='center', va='bottom', fontsize=FS_ANNOT_SM)
    ax.set_xticks(range(len(parts))); ax.set_xticklabels(parts.index)
    ax.set_xlabel('k (0 = BIAS10 only)'); ax.set_title(f'N = {n} control genes')
axes[0, 0].set_ylabel('HC expression variance (adjusted R2, %)')
axes[0, -1].legend(frameon=False, fontsize=FS_ANNOT_SM, loc='upper left', bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(FIG_DIR / 'covariate_4_rda.png', dpi=300, bbox_inches='tight')
plt.show()

,N,k,W,bias_W,bias,batch,batch_left,W_unique,bias_unique
0,0,0,0.0000,0.3562,0.3562,0.2272,0.0572,0.0000,0.3562
1,100,1,0.0584,0.3759,0.3562,0.2272,0.0526,0.0197,0.3175
2,100,2,0.0883,0.3778,0.3562,0.2272,0.0520,0.0216,0.2895
3,100,3,0.2419,0.3861,0.3562,0.2272,0.0490,0.0299,0.1442
4,100,4,0.2724,0.4030,0.3562,0.2272,0.0385,0.0468,0.1306
5,100,5,0.2987,0.4081,0.3562,0.2272,0.0357,0.0519,0.1094
6,100,6,0.3102,0.4122,0.3562,0.2272,0.0342,0.0561,0.1020
7,100,7,0.3239,0.4192,0.3562,0.2272,0.0295,0.0630,0.0953
8,100,8,0.3367,0.4224,0.3562,0.2272,0.0286,0.0662,0.0857
9,100,9,0.3500,0.4252,0.3562,0.2272,0.0283,0.0690,0.0751


/tmp/ipykernel_540966/2611941814.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4b. Where the BIAS10 unique variance comes from (N = N_MAIN)

Same Y as section 4. For each metric m and k: `unique` = R^2(BIAS10 + W_k) - R^2(BIAS10 without m + W_k), the variance only m explains
(k = 0: BIAS10 alone); `alone` = R^2(m). Right: R^2 of each metric regressed on W_1..k (HC), i.e. how much of the metric W reproduces.

In [28]:
BU_KS = [0, 1, 2, 3, 5, 10]
BU_CACHE = OUT / f'rda_bias_unique_N{N_MAIN}_{TAG}.csv'
bu = pd.read_csv(BU_CACHE) if BU_CACHE.is_file() else None
if bu is None:
    Wh = W[N_MAIN][hc_fit]
    rows = []
    for k in BU_KS:
        full = adj_r2(Yc, np.hstack([Xb, Wh[:, :k]]))
        for j, m in enumerate(config.BIAS_COLUMNS):
            rest = np.hstack([np.delete(Xb, j, axis=1), Wh[:, :k]])
            x = Xb[:, j]
            rows.append(dict(k=k, metric=m, unique=full - adj_r2(Yc, rest), alone=adj_r2(Yc, x[:, None]),
                             metric_r2_on_W=0.0 if k == 0 else 1 - rss(basis(np.column_stack([np.ones(len(x)), Wh[:, :k]])), x[:, None])[0] / (x ** 2).sum()))
    bu = pd.DataFrame(rows)
    bu.to_csv(BU_CACHE, index=False)
display((100 * bu.pivot(index='metric', columns='k', values='unique')).loc[config.BIAS_COLUMNS].round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw=dict(width_ratios=[1.6, 1]))
kk = [k for k in BU_KS]
w = 0.8 / len(kk)
cm = plt.cm.Greys(np.linspace(0.3, 1, len(kk)))
for i, k in enumerate(kk):
    d = bu[bu['k'] == k].set_index('metric').loc[config.BIAS_COLUMNS]
    axes[0].bar(np.arange(len(d)) + (i - (len(kk) - 1) / 2) * w, 100 * d['unique'], w, color=cm[i], label='BIAS10 only' if k == 0 else f'+ W_1..{k}')
axes[0].scatter(np.arange(len(config.BIAS_COLUMNS)), 100 * bu[bu['k'] == 0].set_index('metric').loc[config.BIAS_COLUMNS, 'alone'],
                marker='_', s=300, color='#D55E00', label='metric alone', zorder=3)
axes[0].set_xticks(range(len(config.BIAS_COLUMNS))); axes[0].set_xticklabels(config.BIAS_COLUMNS, rotation=45, ha='right')
axes[0].set_ylabel('Adjusted R2 (%)'); axes[0].set_yscale('symlog', linthresh=1)
axes[0].set_title(f'Unique variance of each BIAS10 metric (N = {N_MAIN})')
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM, ncol=2)
mat = bu[bu['k'] > 0].pivot(index='metric', columns='k', values='metric_r2_on_W').loc[config.BIAS_COLUMNS]
im = axes[1].imshow(mat.values, cmap='Greys', vmin=0, vmax=1, aspect='auto')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        axes[1].text(j, i, f'{mat.values[i, j]:.2f}', ha='center', va='center', fontsize=FS_ANNOT_SM,
                     color='white' if mat.values[i, j] > 0.5 else 'black')
axes[1].set_xticks(range(mat.shape[1])); axes[1].set_xticklabels(mat.columns); axes[1].set_xlabel('k')
axes[1].set_yticks(range(mat.shape[0])); axes[1].set_yticklabels(mat.index, fontsize=FS_ANNOT_SM)
axes[1].set_title('R2 of metric ~ W_1..k (HC)')
fig.colorbar(im, ax=axes[1])
fig.tight_layout()
fig.savefig(FIG_DIR / f'covariate_4b_bias_unique_N{N_MAIN}.png', dpi=300, bbox_inches='tight')
plt.show()

k,0,1,2,3,5,10
metric,,,,,,
log(Total Reads),0.75,0.81,0.72,0.65,0.30,0.21
Spliced Reads (%),0.50,0.51,0.37,0.37,0.36,0.17
gDNA Contamination (Intron/Exon),0.75,0.51,0.51,0.43,0.41,0.28
rRNA Fraction,0.37,0.33,0.28,0.26,0.23,0.16
RNA Degradation (3' Bias),1.68,1.63,0.83,0.86,0.77,0.28
Platelet Score,2.40,1.98,0.66,0.34,0.32,0.29
GC Bias,1.28,1.25,1.14,0.51,0.47,0.31
Gene Length Bias,2.73,2.68,2.21,1.68,1.21,0.73
NG80,3.54,1.80,1.85,1.71,1.44,0.37


/tmp/ipykernel_540966/599138554.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. VIF

VIF = diag of the inverse correlation matrix of the covariates (HC). W columns are orthogonal on the HC by construction,
so VIF > 1 comes from overlap with the other covariates. Table: max VIF over all covariates for every N x k,
for BIAS10 + W_1..k and for the compact set depth + W_1..k. Heatmap: per covariate, N = N_MAIN.

In [29]:
def vif(X, names):
    return pd.Series(np.diag(np.linalg.inv(np.corrcoef(X.T))), index=names)


bias_df = pd.DataFrame(data['X_raw'][hc_fit], columns=config.BIAS_COLUMNS)
depth = bias_df[DEPTH].values
wn = [f'W_{i + 1}' for i in range(K_MAX)]
rows = []
for n in N_CONTROLS:
    Wh = W[n][hc_fit]
    for k in range(0, K_MAX + 1):
        rows.append(dict(N=n, k=k, set='BIAS10 + W', max_vif=vif(np.hstack([bias_df.values, Wh[:, :k]]), config.BIAS_COLUMNS + wn[:k]).max()))
        if k:
            rows.append(dict(N=n, k=k, set='depth + W', max_vif=vif(np.column_stack([depth, Wh[:, :k]]), [DEPTH] + wn[:k]).max()))
vt = pd.DataFrame(rows)
vt.to_csv(OUT / f'vif_max_{TAG}.csv', index=False)
display(vt.pivot_table(index=['set', 'N'], columns='k', values='max_vif').round(2))

Wm = W[N_MAIN][hc_fit]
detail = {name: pd.DataFrame({k: vif(np.hstack([base, Wm[:, :k]]), names + wn[:k]) for k in range(k0, K_MAX + 1)}).loc[names + wn]
          for name, base, names, k0 in [('BIAS10 + W', bias_df.values, config.BIAS_COLUMNS, 0),
                                        ('depth + W', depth[:, None], [DEPTH], 1)]}
corr = pd.DataFrame(np.corrcoef(np.column_stack([bias_df.values, Wm]).T),
                    index=config.BIAS_COLUMNS + wn, columns=config.BIAS_COLUMNS + wn).loc[wn, config.BIAS_COLUMNS]
fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw=dict(width_ratios=[1, 0.8, 1.2]))
for ax, (name, v) in zip(axes[:2], detail.items()):
    v.columns.name = 'k'
    im = ax.imshow(np.log10(v.values.astype(float)), cmap='Reds', aspect='auto', vmin=0, vmax=1.7)
    ax.set_xticks(range(v.shape[1])); ax.set_xticklabels(v.columns)
    ax.set_yticks(range(v.shape[0])); ax.set_yticklabels(v.index, fontsize=FS_ANNOT_SM)
    for i in range(v.shape[0]):
        for j in range(v.shape[1]):
            if np.isfinite(v.values[i, j]):
                ax.text(j, i, f'{v.values[i, j]:.1f}', ha='center', va='center', fontsize=6)
    ax.set_xlabel('k (W included)'); ax.set_title(f'VIF, {name} (N = {N_MAIN}, colour: log10)')
im = axes[2].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[2].set_xticks(range(corr.shape[1])); axes[2].set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=FS_ANNOT_SM)
axes[2].set_yticks(range(corr.shape[0])); axes[2].set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        axes[2].text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=6)
axes[2].set_title('Pearson r, W vs BIAS10 (HC)')
fig.colorbar(im, ax=axes[2])
fig.tight_layout()
fig.savefig(FIG_DIR / f'covariate_5_vif_N{N_MAIN}.png', dpi=300, bbox_inches='tight')
plt.show()

k                 0      1      2      3      4      5      6      7      8      9      10
set        N                                                                              
BIAS10 + W 100   5.3   6.92   7.01  21.22  21.51  24.04  24.21  24.88  25.89  27.11  27.81
           300   5.3  22.21  25.87  25.90  27.26  28.32  33.52  34.70  36.70  36.72  38.96
           500   5.3  10.28  20.92  25.49  26.61  26.61  30.74  31.32  33.47  33.54  36.50
           1000  5.3   8.99   9.95  16.73  17.28  17.78  18.53  19.31  31.79  34.30  34.31
depth + W  100   NaN   1.07   1.18   1.18   1.20   1.31   1.32   1.33   1.35   1.45   1.49
           300   NaN   1.10   1.10   1.23   1.24   1.37   1.37   1.37   1.40   1.43   1.46
           500   NaN   1.06   1.10   1.11   1.34   1.38   1.38   1.41   1.41   1.45   1.48
           1000  NaN   1.06   1.12   1.12   1.13   1.45   1.48   1.48   1.48   1.54   1.54

/tmp/ipykernel_540966/134035463.py:25: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw=dict(width_ratios=[1, 0.8, 1.2]))
/tmp/ipykernel_540966/134035463.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. RDA of the compact set: depth + W_1..k

Same Y as section 4. Depth (log(Total Reads)) stays because W comes from CPM and carries no depth, while the engine
models raw counts. Sex is carried by W through the sex genes. Stacked bars per N: first bar = BIAS10 alone (reference),
then depth + W_1..k (k = 0..K_MAX).
`unique` = R^2(full) - R^2(full without that block); shared = the rest of R^2(full); number on top = R^2(full),
number of covariates under each bar. `sexscore_unique` = R^2(depth + W_k + Sex Score) - R^2(depth + W_k): close to 0
means W already carries the sex information (plotted in 8c).

In [30]:
CP_CACHE = OUT / f'rda_compact_{TAG}.csv'
cp = pd.read_csv(CP_CACHE) if CP_CACHE.is_file() else None
if cp is None or 'sexscore_vif' not in cp:
    dz = ((depth - depth.mean()) / depth.std())[:, None]
    sz = ((sex_score[hc_fit] - sex_score[hc_fit].mean()) / sex_score[hc_fit].std())[:, None]
    rows = []
    for n in N_CONTROLS:
        Wh = W[n][hc_fit]
        for k in range(0, K_MAX + 1):
            blocks = dict(depth=dz, W=Wh[:, :k])
            full_x = np.hstack(list(blocks.values()))
            full = adj_r2(Yc, full_x)
            uniq = {b: full - adj_r2(Yc, np.hstack([v for c, v in blocks.items() if c != b])) if blocks[b].shape[1] and k else
                    (full if blocks[b].shape[1] else 0.0) for b in blocks}
            rows.append(dict(N=n, k=k, n_vars=full_x.shape[1], full=full, **{f'{b}_unique': u for b, u in uniq.items()},
                             batch_left=adj_r2(Yc, np.hstack([full_x, Bt])) - full,
                             sexscore_unique=adj_r2(Yc, np.hstack([full_x, sz])) - full,
                             sexscore_vif=vif(np.hstack([full_x, sz]), [f'x{i}' for i in range(full_x.shape[1])] + ['s'])['s']))
    cp = pd.DataFrame(rows)
    cp.to_csv(CP_CACHE, index=False)
display(cp.set_index(['N', 'k']).round(4))

ref = rda[rda['N'] == 0].iloc[0]
CPARTS = [('depth unique', '#E69F00'), ('W unique', '#0072B2'), ('shared', '#999999'),
          ('BIAS10 (reference)', '#D55E00'), ('batch left over', '#F0E442'), ('unexplained', '#E5E5E5')]
fig, axes = plt.subplots(1, len(N_CONTROLS), figsize=(4.2 * len(N_CONTROLS), 4.3), sharey=True, squeeze=False)
for ax, n in zip(axes[0], N_CONTROLS):
    d = cp[cp['N'] == n].set_index('k')
    parts = pd.DataFrame({'depth unique': d['depth_unique'], 'W unique': d['W_unique']})
    parts['shared'] = (d['full'] - parts.sum(1)).clip(lower=0)
    parts['BIAS10 (reference)'] = 0.0
    parts['batch left over'] = d['batch_left']
    parts.loc[-1] = 0.0
    parts.loc[-1, ['BIAS10 (reference)', 'batch left over']] = [ref['bias'], ref['batch_left']]
    parts = parts.sort_index()
    parts['unexplained'] = 1 - parts.sum(1)
    bottom = np.zeros(len(parts))
    for name, color in CPARTS:
        ax.bar(range(len(parts)), 100 * parts[name], bottom=bottom, color=color, edgecolor='white', lw=0.4, label=name)
        bottom += 100 * parts[name].values
    tops = [ref['bias']] + list(d['full'])
    lefts = [ref['batch_left']] + list(d['batch_left'])
    for x, (v, b) in enumerate(zip(tops, lefts)):
        ax.text(x, 100 * (v + b) + 1, f'{100 * v:.0f}', ha='center', va='bottom', fontsize=FS_ANNOT_SM)
    ax.set_xticks(range(len(parts)))
    ax.set_xticklabels(['BIAS10\n(10)'] + [f'{k}\n({nv})' for k, nv in zip(d.index, d['n_vars'])], fontsize=FS_ANNOT_SM)
    ax.set_xlabel('k (number of covariates)'); ax.set_title(f'depth + W_1..k, N = {n}')
axes[0, 0].set_ylabel('HC expression variance (adjusted R2, %)')
axes[0, -1].legend(frameon=False, fontsize=FS_ANNOT_SM, loc='upper left', bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(FIG_DIR / 'covariate_6_rda_compact.png', dpi=300, bbox_inches='tight')
plt.show()

n_vars    full  depth_unique  W_unique  batch_left  sexscore_unique  sexscore_vif
N    k                                                                                    
100  0        1  0.0211        0.0211    0.0000      0.2248           0.0089        1.0741
     1        2  0.0758        0.0174    0.0548      0.1936           0.0081        1.0859
     2        3  0.0970        0.0087    0.0759      0.1781           0.1432       41.2415
     3        4  0.2497        0.0078    0.2286      0.1099           0.0073      310.4673
     4        5  0.2790        0.0066    0.2579      0.0915           0.0094      324.9967
     5        6  0.3039        0.0051    0.2828      0.0860           0.0094      375.4427
     6        7  0.3155        0.0052    0.2944      0.0801           0.0087      386.4057
     7        8  0.3292        0.0053    0.3082      0.0685           0.0093      393.0211
     8        9  0.3415        0.0048    0.3204      0.0598           0.0066      511.8761
     9       10  0.3538        0.0038    0.3327      0.0522           0.0068      512.1359
     10      11  0.3645        0.0033    0.3434      0.0488           0.0033     1012.6383
300  0        1  0.0211        0.0211    0.0000      0.2248           0.0089        1.0741
     1        2  0.1078        0.0090    0.0867      0.1945           0.0091        1.0741
     2        3  0.2603        0.0091    0.2392      0.1097           0.0038        1.1545
     3        4  0.2661        0.0060    0.2451      0.1073           0.0138       27.1183
     4        5  0.2972        0.0054    0.2761      0.0883           0.0143       27.1416
     5        6  0.3155        0.0047    0.2944      0.0813           0.0166       35.8525
     6        7  0.3334        0.0048    0.3123      0.0740           0.0142       69.3081
     7        8  0.3521        0.0047    0.3310      0.0617           0.0079       95.9967
     8        9  0.3621        0.0042    0.3411      0.0550           0.0076      207.1877
     9       10  0.3732        0.0039    0.3521      0.0473           0.0087      215.5421
     10      11  0.3973        0.0037    0.3762      0.0359           0.0051      310.8025
500  0        1  0.0211        0.0211    0.0000      0.2248           0.0089        1.0741
     1        2  0.2002        0.0114    0.1791      0.1543           0.0047        1.0879
     2        3  0.2680        0.0092    0.2470      0.1096           0.0032        1.1240
     3        4  0.3026        0.0085    0.2815      0.0883           0.0027        1.1413
     4        5  0.3141        0.0041    0.2930      0.0798           0.0109        3.3929
     5        6  0.3321        0.0043    0.3110      0.0722           0.0114        4.6011
     6        7  0.3466        0.0043    0.3255      0.0690           0.0115       24.4241
     7        8  0.3628        0.0042    0.3417      0.0578           0.0066       42.4216
     8        9  0.3762        0.0043    0.3552      0.0499           0.0158       74.3679
     9       10  0.3858        0.0037    0.3648      0.0432           0.0171       78.2763
     10      11  0.4114        0.0034    0.3903      0.0296           0.0045      167.1006
1000 0        1  0.0211        0.0211    0.0000      0.2248           0.0089        1.0741
     1        2  0.2190        0.0116    0.1980      0.1476           0.0039        1.0957
     2        3  0.2876        0.0077    0.2665      0.0983           0.0033        1.1118
     3        4  0.3147        0.0076    0.2936      0.0865           0.0024        1.1638
     4        5  0.3339        0.0078    0.3128      0.0808           0.0026        1.2202
     5        6  0.3517        0.0039    0.3306      0.0676           0.0025        1.4641
     6        7  0.3816        0.0038    0.3605      0.0504           0.0101        2.0399
     7        8  0.3940        0.0039    0.3729      0.0436           0.0095        4.4076
     8        9  0.4063        0.0039    0.3853      0.0371           0.0068        8.1869
     

/tmp/ipykernel_540966/3041354615.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. In-silico projection absorption

Does W take up injected signal? Frozen in-silico cells (`eval/insilico_sweep.py`, 500 random universe genes x log2FC,
every held-out HC of each LOBO batch / CV fold). Per fold, W is fitted on that fold's training HC control genes and
projected onto the injected cells (library size updated with the injection).
Linear proxy of the engine: per gene OLS of log2 CPM fitted on the training HC, with the compact set [1, depth, W_1..k]
(`depth + W`, the p2 candidate, solid lines) and for [1, BIAS10, W_1..k] (`BIAS10 + W`, comparison, dashed lines).

- `absorbed`: share of the injected log2 CPM shift that the W term ((Delta W) beta_W) explains away (negative = W amplifies)
- `dW_hc_sd`: mean |Delta W| in units of the training-HC SD of W
- `spill_pct`: mean |(Delta W) beta_W| on non-injected genes, as % of the mean injected shift (both in residual-SD units)

Depth and BIAS10 are fixed in the frozen design (in real data NG80, Platelet Score etc. would also move with the
injection), so only W can absorb here; the comparison is conservative against W.

In [31]:
ABS_KS = [1, 2, 3, 5, 6, 10]
ABS_MODELS = {'depth + W': [config.BIAS_COLUMNS.index(DEPTH)], 'BIAS10 + W': list(range(len(config.BIAS_COLUMNS)))}
ABS_STYLE = {'depth + W': '-', 'BIAS10 + W': '--'}
ABS_LOG2FCS = [1.0, 2.0, 3.0, 5.0]
ABS_CACHE = OUT / f'absorption_summary_{TAG}_depthW_biasW_k{"-".join(map(str, ABS_KS))}.csv'
absorb = pd.read_csv(ABS_CACHE) if ABS_CACHE.is_file() else None
if absorb is None:
    U = universe()
    u_idx = np.array([col[g] for g in U])
    row_of = {s: i for i, s in enumerate(data['names'])}
    ctrl_idx = {n: np.array([col[g] for g in c]) for n, c in controls.items()}
    rows = []
    for split in ('lobo', 'cv'):
        for key, tag, tr, te in fold_spec(split):
            tr_r, te_r = np.array([row_of[s] for s in tr]), np.array([row_of[s] for s in te])
            m, sd = data['X_raw'][tr_r].mean(0), data['X_raw'][tr_r].std(0)
            Xb_tr = (data['X_raw'][tr_r] - m) / sd
            Lu_tr = logcpm[tr_r][:, u_idx]
            cells = {f: np.load(cell_path(split, tag, f)) for f in [0.0] + ABS_LOG2FCS}
            pert = cells[1.0]['labels']
            test_L = {}
            for f, c in cells.items():
                counts = Y[te_r].copy()
                counts[:, u_idx] = c['y_pert']
                lib_f = lib[te_r] + (c['y_pert'] - Y[te_r][:, u_idx]).sum(1)
                test_L[f] = np.log2(counts / lib_f[:, None] * 1e6 + 1)
            for n, ci in ctrl_idx.items():
                fit = ruvg_fit(logcpm[tr_r][:, ci], max(ABS_KS))
                W_tr = ruvg_transform(fit, logcpm[tr_r][:, ci])
                W_te = {f: ruvg_transform(fit, x[:, ci]) for f, x in test_L.items()}
                for model, cols in ABS_MODELS.items():
                    for k in ABS_KS:
                        X_tr = np.column_stack([np.ones(len(tr_r)), Xb_tr[:, cols], W_tr[:, :k]])
                        beta = np.linalg.lstsq(X_tr, Lu_tr, rcond=None)[0]
                        res_sd = (Lu_tr - X_tr @ beta).std(0) + 1e-9
                        for f in ABS_LOG2FCS:
                            A = (W_te[f][:, :k] - W_te[0.0][:, :k]) @ beta[-k:]
                            dy = test_L[f][:, u_idx] - test_L[0.0][:, u_idx]
                            signal = (np.abs(dy[:, pert]) / res_sd[pert]).mean()
                            rows.append(dict(split=split, fold=key, model=model, N=n, k=k, log2fc=f,
                                             absorbed=A[:, pert].sum() / dy[:, pert].sum(),
                                             dW_hc_sd=(np.abs(W_te[f][:, :k] - W_te[0.0][:, :k]) / W_tr[:, :k].std(0)).mean(),
                                             spill_pct=100 * (np.abs(A[:, ~pert]) / res_sd[~pert]).mean() / signal))
            print(f'absorption {split}/{tag}', flush=True)
    absorb = pd.DataFrame(rows).groupby(['split', 'model', 'N', 'k', 'log2fc']).mean(numeric_only=True).reset_index()
    absorb.to_csv(ABS_CACHE, index=False)
print(f'control-set sizes N = {N_CONTROLS} (+ sex genes; least phenotype-associated first; {len(cand)} genes pass DETECT_MIN)')
display(absorb.pivot_table(index=['split', 'model', 'N', 'k'], columns='log2fc', values=['absorbed', 'spill_pct']).round(3))

KCOL = dict(zip(ABS_KS, ['#E69F00', '#56B4E9', '#009E73', '#CC79A7', '#0072B2', '#000000']))
for split in ('lobo', 'cv'):
    fig, axes = plt.subplots(3, len(N_CONTROLS), figsize=(3.3 * len(N_CONTROLS), 9), sharex=True, sharey='row', squeeze=False)
    for j, n in enumerate(N_CONTROLS):
        for (model, k), d in absorb[(absorb['split'] == split) & (absorb['N'] == n)].groupby(['model', 'k']):
            kw = dict(marker='o', ms=3, ls=ABS_STYLE[model], color=KCOL[k])
            axes[0, j].plot(d['log2fc'], 100 * d['absorbed'], label=f'k={k}' if model == 'depth + W' else None, **kw)
            axes[1, j].plot(d['log2fc'], d['dW_hc_sd'], **kw)
            axes[2, j].plot(d['log2fc'], d['spill_pct'], **kw)
        axes[0, j].axhline(0, color='0.5', lw=0.8)
        axes[0, j].set_title(f'N = {n}')
        axes[2, j].set_xlabel('Injected log2FC')
    axes[0, 0].set_ylabel('Injected signal absorbed by W (%)')
    axes[1, 0].set_ylabel('|Delta W| (HC SD units)')
    axes[2, 0].set_ylabel('Spillover on non-injected genes\n(% of injected signal)')
    for model, ls in ABS_STYLE.items():
        axes[0, -1].plot([], [], color='0.4', ls=ls, label=model)
    axes[0, -1].legend(frameon=False, fontsize=FS_ANNOT_SM)
    fig.suptitle(f'In-silico projection absorption, {split} (linear proxy)')
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'covariate_7_absorption_{split}.png', dpi=300, bbox_inches='tight')
    plt.show()

control-set sizes N = [100, 300, 500, 1000] (+ sex genes; least phenotype-associated first; 1227 genes pass DETECT_MIN)


absorbed                      spill_pct                      
log2fc                        1.0    2.0    3.0    5.0       1.0    2.0    3.0     5.0
split model      N    k                                                               
cv    BIAS10 + W 100  1    -0.000 -0.006 -0.013 -0.039     1.313  1.686  2.997   8.905
                      2    -0.000 -0.006 -0.013 -0.039     1.320  1.705  3.036   9.011
                      3    -0.003 -0.005 -0.008 -0.019     2.182  2.371  3.374   8.330
                      5    -0.002 -0.003 -0.006 -0.017     3.629  3.758  4.558   9.075
                      6    -0.002 -0.003 -0.006 -0.015     3.720  3.854  4.706   9.492
...                           ...    ...    ...    ...       ...    ...    ...     ...
lobo  depth + W  1000 2     0.006 -0.007 -0.028 -0.103     1.639  1.842  3.278  10.731
                      3     0.007 -0.009 -0.033 -0.119     1.749  2.176  3.969  12.619
                      5     0.007 -0.008 -0.032 -0.119     1.851  2.233  4.030  12.848
                      6     0.008 -0.009 -0.036 -0.132     1.934  2.346  4.277  13.802
                      10    0.009 -0.008 -0.035 -0.129     2.279  2.653  4.482  13.671

[96 rows x 8 columns]

/tmp/ipykernel_540966/2122640770.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_540966/2122640770.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Choosing N and k

Every criterion used for the control-set size N and the number of factors k, one figure each, all from the tables of
sections 1-7 (no new fits). Colour = N (`N_CONTROLS`) or k; dotted line = the current choice (`N_MAIN`, `K_MAIN`).

| fig | criterion | rule |
|---|---|---|
| 8a | scree of the HC control matrix | k: where extra components add little control-gene variance |
| 8b | bootstrap stability of W_1..k | N >= `N_LOWER`; k: subspace reproducible (5th pct >= `STAB_MIN`) |
| 8c | sex carried by W | R^2 of known sex / Sex Score on W high; Sex Score adds ~0 unique R^2 beyond depth + W |
| 8d | explained HC variance | R^2(depth + W_1..k) not below BIAS10; batch left over not above BIAS10 |
| 8e | VIF | max VIF of depth + W_1..k low (BIAS10 + W for comparison) |
| 8f | in-silico absorption / spillover (LOBO) | small absorption of injected signal, small spill onto other genes |
| 8g | phenotype association of the control set | N: larger sets take in more phenotype-associated genes |

In [32]:
NCOL = dict(zip(N_CONTROLS, ['#E69F00', '#56B4E9', '#0072B2', '#000000']))
bias_ref = rda[rda['N'] == 0].iloc[0]


def mark(ax, x):
    ax.axvline(x, color='0.5', lw=0.8, ls=':')


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'covariate_8{name}.png', dpi=300, bbox_inches='tight')
    plt.show()


fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for n, fit in fits.items():
    axes[0].plot(range(1, K_MAX + 1), 100 * fit['var_frac'][:K_MAX], marker='o', ms=3, color=NCOL[n], label=f'N={n}')
    axes[1].plot(range(1, K_MAX + 1), 100 * np.cumsum(fit['var_frac'][:K_MAX]), marker='o', ms=3, color=NCOL[n])
axes[0].set_ylabel('% of control-gene variance (HC)'); axes[1].set_ylabel('Cumulative % (W_1..k)')
for ax in axes:
    ax.set_xlabel('k'); mark(ax, K_MAIN)
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.suptitle('8a. Scree of the HC control matrix')
save(fig, 'a_scree')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
fig, ax = plt.subplots(figsize=(4.8, 3.6))
for k in STAB_KS:
    ax.plot(stab_summary.index, stab_summary[('50%', k)], marker='o', ms=3, color=KCOL[k], label=f'k={k}')
    ax.fill_between(stab_summary.index, stab_summary[('5%', k)], stab_summary[('50%', k)], color=KCOL[k], alpha=0.2, lw=0)
ax.axhline(STAB_MIN, color='0.5', lw=0.8, ls='--')
if N_LOWER is not None:
    ax.axvline(N_LOWER, color='#D55E00', lw=0.8, ls='--', label=f'N_LOWER = {N_LOWER}')
mark(ax, N_MAIN)
ax.set_xscale('log'); ax.set_xticks(STAB_NS); ax.set_xticklabels(STAB_NS, rotation=45)
ax.set_xlabel('Control-set size N (+ sex genes)'); ax.set_ylabel('Mean canonical correlation\n(bootstrap vs full HC fit; band 5-50%)')
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
ax.set_title('8b. Bootstrap stability of W')
save(fig, 'b_stability')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [34]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for n in N_CONTROLS:
    kw = dict(marker='o', ms=3, color=NCOL[n])
    axes[0].plot(sex_w.index, sex_w[n], label=f'N={n}', **kw)
    axes[1].plot(score_w.index, score_w[n], **kw)
    d = cp[cp['N'] == n]
    axes[2].plot(d['k'], 100 * d['sexscore_unique'], **kw)
    axes[3].plot(d['k'], d['sexscore_vif'], **kw)
axes[0].set_ylabel('R2, known sex ~ W_1..k\n(known-sex HC)'); axes[1].set_ylabel('R2, Sex Score ~ W_1..k (HC)')
axes[2].set_ylabel('Sex Score unique R2\nbeyond depth + W_1..k (%)'); axes[3].set_ylabel('VIF of Sex Score\nadded to depth + W_1..k')
axes[3].set_yscale('log')
for ax in axes:
    ax.set_xlabel('k'); mark(ax, K_MAIN)
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.suptitle('8c. Does W carry sex?')
save(fig, 'c_sex')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [35]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for n in N_CONTROLS:
    d = cp[cp['N'] == n]
    axes[0].plot(d['k'], 100 * d['full'], marker='o', ms=3, color=NCOL[n], label=f'N={n}')
    axes[1].plot(d['k'], 100 * d['batch_left'], marker='o', ms=3, color=NCOL[n])
axes[0].axhline(100 * bias_ref['bias'], color='#D55E00', lw=0.8, ls='--', label='BIAS10 (10 covariates)')
axes[1].axhline(100 * bias_ref['batch_left'], color='#D55E00', lw=0.8, ls='--')
axes[0].set_ylabel('HC variance explained by depth + W_1..k\n(adjusted R2, %)')
axes[1].set_ylabel('Batch variance left over (%)')
for ax in axes:
    ax.set_xlabel('k (covariates = k + 1)'); mark(ax, K_MAIN)
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.suptitle('8d. Explained HC variance, compact set vs BIAS10')
save(fig, 'd_explained_variance')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
fig, ax = plt.subplots(figsize=(4.8, 3.6))
for (s, n), d in vt.groupby(['set', 'N']):
    d = d[d['k'] > 0]
    ax.plot(d['k'], d['max_vif'], marker='o', ms=3, color=NCOL[n], ls='-' if s == 'depth + W' else '--',
            label=f'N={n}' if s == 'depth + W' else None)
for s, ls in [('depth + W', '-'), ('BIAS10 + W', '--')]:
    ax.plot([], [], color='0.4', ls=ls, label=s)
ax.set_yscale('log'); ax.set_xlabel('k'); ax.set_ylabel('Max VIF over covariates (HC)')
mark(ax, K_MAIN)
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
ax.set_title('8e. VIF')
save(fig, 'e_vif')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
fcs = [min(ABS_LOG2FCS), max(ABS_LOG2FCS)]
a = absorb[(absorb['split'] == 'lobo') & (absorb['model'] == 'depth + W')]
fig, axes = plt.subplots(2, len(fcs), figsize=(4.2 * len(fcs), 6.5), sharex=True, sharey='row', squeeze=False)
for j, f in enumerate(fcs):
    for k, d in a[a['log2fc'] == f].groupby('k'):
        kw = dict(marker='o', ms=3, color=KCOL[k], lw=2 if k == K_MAIN else 1)
        axes[0, j].plot(d['N'], 100 * d['absorbed'], label=f'k={k}', **kw)
        axes[1, j].plot(d['N'], d['spill_pct'], **kw)
    axes[0, j].axhline(0, color='0.5', lw=0.8)
    axes[0, j].set_title(f'log2FC = {f:g}')
    axes[1, j].set_xscale('log'); axes[1, j].set_xticks(N_CONTROLS); axes[1, j].set_xticklabels(N_CONTROLS, rotation=45)
    axes[1, j].set_xlabel('Control-set size N')
    for ax in axes[:, j]:
        mark(ax, N_MAIN)
axes[0, 0].set_ylabel('Injected signal absorbed by W (%)')
axes[1, 0].set_ylabel('Spillover on non-injected genes\n(% of injected signal)')
axes[0, -1].legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.suptitle('8f. In-silico absorption, LOBO, depth + W (linear proxy)')
save(fig, 'f_absorption')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
pa = pd.DataFrame({n: dict(median=stats.loc[cand.index[:n], 'pheno_r2'].median(), max=stats.loc[cand.index[:n], 'pheno_r2'].max())
                   for n in STAB_NS}).T
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.plot(pa.index, 100 * pa['median'], marker='o', ms=3, color='#0072B2', label='median')
ax.plot(pa.index, 100 * pa['max'], marker='o', ms=3, color='#000000', label='max')
ax.set_xscale('log'); ax.set_xticks(STAB_NS); ax.set_xticklabels(STAB_NS, rotation=45)
ax.set_xlabel('Control-set size N (sex genes excluded)'); ax.set_ylabel('Phenotype partial R2 in the set (%)')
mark(ax, N_MAIN)
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
ax.set_title('8g. Phenotype association of the control set')
save(fig, 'g_phenotype_association')

/tmp/ipykernel_540966/3291408762.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8h. Summary at the current choice (`N_MAIN`, `K_MAIN`)

In [39]:
c = cp[(cp['N'] == N_MAIN) & (cp['k'] == K_MAIN)].iloc[0]
v = vt.set_index(['set', 'N', 'k'])['max_vif']
ab = a[(a['N'] == N_MAIN) & (a['k'] == K_MAIN)].set_index('log2fc')
choice = pd.DataFrame([
    ('8a', 'cumulative control-gene variance of W_1..k (%)', 100 * fits[N_MAIN]['var_frac'][:K_MAIN].sum()),
    ('8b', 'stability, 5th pct of mean canonical correlation', stab_summary.loc[N_MAIN, ('5%', K_MAIN)]),
    ('8b', 'N_LOWER', N_LOWER),
    ('8c', 'R2 known sex ~ W', sex_w.loc[K_MAIN, N_MAIN]),
    ('8c', 'R2 Sex Score ~ W', score_w.loc[K_MAIN, N_MAIN]),
    ('8c', 'Sex Score unique R2 beyond depth + W (%)', 100 * c['sexscore_unique']),
    ('8c', 'VIF of Sex Score', c['sexscore_vif']),
    ('8d', 'adjusted R2, depth + W (%)', 100 * c['full']),
    ('8d', 'adjusted R2, BIAS10 (%)', 100 * bias_ref['bias']),
    ('8d', 'batch left over, depth + W (%)', 100 * c['batch_left']),
    ('8d', 'batch left over, BIAS10 (%)', 100 * bias_ref['batch_left']),
    ('8e', 'max VIF, depth + W', v[('depth + W', N_MAIN, K_MAIN)]),
    ('8e', 'max VIF, BIAS10 + W', v[('BIAS10 + W', N_MAIN, K_MAIN)]),
    *[('8f', f'{m}, log2FC {f:g} (%)', (100 if m == 'absorbed' else 1) * ab.loc[f, m]) for m in ('absorbed', 'spill_pct') for f in fcs],
    ('8g', 'max phenotype partial R2 in the set (%)', 100 * pa.loc[N_MAIN, 'max']),
], columns=['figure', 'evidence', 'value'])
choice.to_csv(OUT / f'choice_summary_N{N_MAIN}_k{K_MAIN}_{TAG}.csv', index=False)
display(choice.round(3))

,figure,evidence,value
0,8a,cumulative control-gene variance of W_1..k (%),69.454
1,8b,"stability, 5th pct of mean canonical correlation",0.982
2,8b,N_LOWER,100.000
3,8c,R2 known sex ~ W,0.916
4,8c,R2 Sex Score ~ W,0.957
5,8c,Sex Score unique R2 beyond depth + W (%),1.151
6,8c,VIF of Sex Score,24.424
7,8d,"adjusted R2, depth + W (%)",34.658
8,8d,"adjusted R2, BIAS10 (%)",35.619
9,8d,"batch left over, depth + W (%)",6.896
